In [2]:
# Install necessary libraries
%pip install openai tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import json
import time
from openai import OpenAI
from tqdm import tqdm

In [ ]:
# --- CONFIGURATION ---
# Path to the output file based on your image structure
output_path = "../../data/processed/augmented_synthetic_500.jsonl"

# Replace with your actual OpenAI API Key
api_key = "YOUR_OPENAI_API_KEY"

client = OpenAI(api_key=api_key)

In [5]:
# System prompt targeting "Hard Negatives" to address weaknesses
system_prompt = """
You are an expert creative writer and logic puzzle generator. 
Your goal is to generate "Narrative Cloze" tasks that require Chain-of-Thought reasoning.

Structure:
1. InputStory: A short story (4 sentences).
2. Two potential endings (Option 1 and Option 2).
3. The correct answer (1 or 2).

CRITICAL REQUIREMENT:
You must generate "Hard Negatives". 
- The incorrect ending must be THEMATICALLY related to the story but LOGICALLY incorrect based on the sequence of events.
- Do not make the wrong answer obvious. It should trick a simple model, but be solvable by a human paying attention to causal logic.
"""

user_prompt_template = """
Generate a JSON object with the following structure:
{
  "InputStoryid": "unique_id",
  "InputSentence1": "First sentence.",
  "InputSentence2": "Second sentence.",
  "InputSentence3": "Third sentence.",
  "InputSentence4": "Fourth sentence.",
  "RandomFifthSentenceQuiz1": "Option 1.",
  "RandomFifthSentenceQuiz2": "Option 2.",
  "AnswerRightEnding": 1 or 2
}
"""

In [ ]:
def generate_synthetic_data(target_count=550):
    # Ensure the directory exists
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    print(f"Starting generation. Saving to: {output_path}")
    
    generated_count = 0
    
    # Open file in append mode
    with open(output_path, 'a', encoding='utf-8') as f:
        for i in tqdm(range(target_count)):
            try:
                response = client.chat.completions.create(
                    model="gpt-5.1", 
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt_template}
                    ],
                    temperature=0.8,
                    response_format={"type": "json_object"}
                )
                
                content = response.choices[0].message.content
                data = json.loads(content)
                
                # Add unique ID
                data['InputStoryid'] = f"aug_syn_{i}"
                
                # Write immediately to file
                f.write(json.dumps(data) + "\n")
                generated_count += 1
                
            except Exception as e:
                print(f"Error at index {i}: {e}")
                continue
                
    print(f"Finished! Generated {generated_count} samples.")

# Run the function
generate_synthetic_data()

Starting generation. Saving to: ../../data/processed/augmented_synthetic_500.jsonl


100%|██████████| 550/550 [2:29:44<00:00, 16.34s/it]     

Finished! Generated 550 samples.
